# Train an SPD of GPT-2 small on a Colab GPU

This notebook runs `scripts/train_spd.py` (Stochastic Parameter Decomposition of GPT-2 small's MLP projections) for longer than a laptop can. The runs so far are in `docs/analyses/spd-baseline.md`.

**Runtime.** *Runtime → Change runtime type → A100 GPU* (Colab Pro). A T4 is too slow.

**Before you start.** Upload `corpus.jsonl` from the repository root to `MyDrive/openinterp/corpus.jsonl`. The trainer leaves its documents out of the training data. The corpus is not in git because of its licence.

**Runs.** Run each in its own folder (`RUN`):

1. Run A (done): `MATRICES = "down"`, `INIT = "random"`. The down projections only, for 50,000 steps.
2. Run B (done): `MATRICES = "down"`, `INIT = "neurons"`. One subcomponent per neuron at the start (V = I, U = W).
3. Run C: `MATRICES = "both"`, `INIT = "random"`, `RUN = "gpt2-spd-both-random-50k"`. The input projections (`c_fc`) too. GELU acts on each neuron separately, so there SPD can favour subcomponents that are neurons. Do the input projection's subcomponents turn towards the neurons from a random start?
4. Run D: `MATRICES = "both"`, `INIT = "neurons"`, `RUN = "gpt2-spd-both-neurons-50k"`. The same from the neurons. Does it fit the input projections better than run C at equal sparsity?

**If the runtime disconnects,** run every cell again with the same `RUN`. Training continues from the last checkpoint in Drive (every 1,000 steps), with a new optimizer.

**When it finishes,** send back `compare.json` and `train.log` from the run's Drive folder. They are small. Keep `checkpoint.pt` and the `layer_*.safetensors` files in Drive.

In [ ]:
BRANCH = "main"
RUN = "gpt2-spd-both-random-50k"  # the folder in MyDrive/openinterp; one per run
MATRICES = "both"  # "down", "up", or "both" (the MLP's down and input projections)
INIT = "random"  # "random", or "neurons" for one subcomponent per neuron
STEPS = 50_000
EXTRA = []  # more train_spd.py options, e.g. ["--batch-rows", "16"]

In [ ]:
import pathlib
import shlex

from google.colab import drive

drive.mount("/content/drive")
ROOT = pathlib.Path("/content/drive/MyDrive/openinterp")
OUT = ROOT / RUN
OUT.mkdir(parents=True, exist_ok=True)
CORPUS = ROOT / "corpus.jsonl"
assert CORPUS.exists(), f"Upload corpus.jsonl from the repository root to {CORPUS}"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!rm -rf openinterpretability
!git clone --depth 1 --branch {BRANCH} https://github.com/8W9aG/openinterpretability.git
%cd /content/openinterpretability
!pip install -q -e .

Training prints one line every 1,000 steps, with the hours left. Per site, `l0` is the active subcomponents per token and `explained_variance` the fit of the CI-masked projection. A site is a layer's down projection (`"5"`) or its input projection (`"5.up"`, measured after GELU). Run A ended at 225–695 active subcomponents and a fit of 0.84–0.97 on the down projections.

In [ ]:
args = [
    "--layers", "all", "--steps", str(STEPS), "--eval-every", "1000",
    "--matrices", MATRICES, "--gate", "layer", "--d-gate", "512", "--init", INIT,
    "--coeff-imp", "6e-5", "--coeff-ci-output", "10", "--center", "--p-anneal", "0.1",
    "--imp-eps", "1e-3", "--gate-grad-clip", "1.0",
    "--scan-corpus", str(CORPUS), "--out", str(OUT), *EXTRA,
]  # fmt: skip
if (OUT / "checkpoint.pt").exists():
    args += ["--resume", str(OUT / "checkpoint.pt")]
train = shlex.join(["python", "train_spd.py", *args])
log = shlex.quote(str(OUT / "train.log"))
!cd scripts && {train} 2>&1 | tee -a {log}

Compare the result with the native neurons on held-out text: density at equal fit, alignment with the neuron axes, and drift from initialisation (see `scripts/spd_compare.py`). For an input projection, alignment and drift read each subcomponent's output direction, `U[c]`.

In [ ]:
compare = shlex.join([
    "python", "spd_compare.py", "--checkpoint", str(OUT / "checkpoint.pt"),
    "--scan-corpus", str(CORPUS), "--out", str(OUT / "compare.json"),
])  # fmt: skip
!cd scripts && {compare}